# 04 — Neural Nets & CNNs with PyTorch (Phase 4, Milestone 4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/udaysharmadev/Ai-Roadmap/blob/main/notebooks/04_pytorch_cnn.ipynb)

**Maps to:** `docs/deep-learning-roadmap.md > Phase 4` + `README Milestone 4`  
**Data:** synthetic quadrant digits (`make_synthetic_digits`, no download, CPU-fast)  
**Goal:** train a custom CNN classifier — forward pass → loss → backprop → evaluate → save.

MLP baseline vs SimpleCNN, 5 epochs, <2 min on laptop CPU.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch

from ai_roadmap.torch_utils import (
    SimpleCNN,
    SimpleMLP,
    count_parameters,
    evaluate_accuracy,
    get_device,
    make_loaders,
    make_synthetic_digits,
    save_model,
    set_seed,
    train_classifier,
)

set_seed(42)
device = get_device()
print(f"device: {device} | torch: {torch.__version__}")

## 1. Synthetic digits — look at the task

Each class lights up one quadrant + a class bar, plus noise. Trivial for a CNN, hard for a blind guess (25%).

In [ ]:
ds = make_synthetic_digits(200, img_size=16, num_classes=4, seed=42)
print(f"dataset: {len(ds)} images")
x0, y0 = ds[0]
print(f"sample: {tuple(x0.shape)} label={int(y0)}")

fig, axes = plt.subplots(1, 4, figsize=(8, 2.5))
for cls in range(4):
    idx = (ds.tensors[1] == cls).nonzero()[0].item()
    axes[cls].imshow(ds.tensors[0][idx, 0], cmap="gray", vmin=0, vmax=1)
    axes[cls].set_title(f"class {cls}")
    axes[cls].axis("off")
fig.suptitle("Synthetic digits — one sample per class")
fig.tight_layout()
out = ROOT / "outputs"
out.mkdir(exist_ok=True)
fig.savefig(out / "synthetic_digits.png", dpi=120)
plt.close(fig)
print(f"saved: {out / 'synthetic_digits.png'}")

## 2. Models: MLP baseline vs tiny CNN

MLP flattens pixels (no spatial prior). CNN uses convolutions + pooling — the inductive bias behind Milestone 4.

In [ ]:
mlp = SimpleMLP(input_dim=16 * 16, hidden_dims=(128, 64), num_classes=4)
cnn = SimpleCNN(num_classes=4)
print(f"MLP params: {count_parameters(mlp):,}")
print(f"CNN params: {count_parameters(cnn):,}")
print(f"MLP out: {tuple(mlp(torch.randn(2, 1, 16, 16)).shape)}")
print(f"CNN out: {tuple(cnn(torch.randn(2, 1, 16, 16)).shape)}")

## 3. Train both (5 epochs, CPU)

In [ ]:
train_loader, val_loader = make_loaders(ds, batch_size=64, seed=42)
print(f"batches: {len(train_loader)} train / {len(val_loader)} val")

set_seed(42)
hist_mlp = train_classifier(
    SimpleMLP(input_dim=256, num_classes=4),
    train_loader, val_loader, epochs=5, lr=5e-3, device=device,
)
set_seed(42)
cnn_model = SimpleCNN(num_classes=4)
hist_cnn = train_classifier(cnn_model, train_loader, val_loader, epochs=5, lr=5e-3, device=device)

print(f"MLP val acc: {hist_mlp['val_acc'][-1]:.3f}")
print(f"CNN val acc: {hist_cnn['val_acc'][-1]:.3f}")

## 4. Plot learning curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(hist_mlp["train_loss"], label="mlp")
ax1.plot(hist_cnn["train_loss"], label="cnn")
ax1.set_title("Train loss")
ax1.set_xlabel("epoch")
ax1.legend()
ax2.plot(hist_mlp["val_acc"], label="mlp")
ax2.plot(hist_cnn["val_acc"], label="cnn")
ax2.set_title("Val accuracy")
ax2.set_xlabel("epoch")
ax2.legend()
fig.tight_layout()
fig.savefig(out / "cnn_curves.png", dpi=120)
plt.close(fig)
print(f"saved: {out / 'cnn_curves.png'}")

final = evaluate_accuracy(cnn_model, val_loader, device)
print(f"final CNN val accuracy: {final:.3f}")
assert final > 0.8, "CNN should exceed 80% on quadrant digits"
print("Milestone-4 check passed ✅")

## 5. Save the checkpoint (Chunk 6 will serve it)

In [ ]:
ckpt = save_model(cnn_model, out / "cnn_digits.pt")
print(f"saved: {ckpt} ({ckpt.stat().st_size / 1024:.1f} KB)")

# Reload sanity check
from ai_roadmap.torch_utils import load_model

fresh = SimpleCNN(num_classes=4)
load_model(fresh, ckpt, device=device)
acc = evaluate_accuracy(fresh, val_loader, device)
print(f"reloaded accuracy: {acc:.3f}")

## ✅ What you learned (interview-ready)

- Forward pass → CrossEntropy loss → backprop → Adam step (see `train_classifier`)
- Why CNNs beat MLPs on images: weight sharing + translation invariance
- ReLU keeps gradients alive (vs sigmoid saturation) — see interview Q4

**Next (Chunk 4):** `05_rag_minimal.ipynb` — embeddings + cosine search over documents.